In [7]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEndpoint ,ChatHuggingFace
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [ ]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"

llm = HuggingFaceEndpoint(
    repo_id=model_id,
    task="text-generation",
    max_new_tokens=256,
    huggingfacehub_api_token="adfas",
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1,
)

model = ChatHuggingFace(llm=llm)

In [20]:
class MyState(TypedDict):
    question: str
    ans: str

In [38]:
def question_llm(state: MyState) -> MyState:
    question = state['question']

    prompt = f"Answer the following question: {question}"

    answer = model.invoke(prompt).content
    # print(answer)

    state['ans'] = answer

    return state

In [22]:
def ask_question(state: MyState) -> MyState:
    # state['question'] = input('What is your query')
    return state

In [39]:
graph = StateGraph(MyState)

graph.add_node('ask_question', ask_question)
graph.add_node('find_answer', question_llm)

graph.add_edge(START, 'ask_question')
graph.add_edge('ask_question', 'find_answer')
graph.add_edge('find_answer', END)

workflow = graph.compile()

In [40]:
initial_state = {'question': 'Who is MS Dhoni?', 'ans': ''}
result = workflow.invoke(initial_state)

print(result)

{'question': 'Who is MS Dhoni?', 'ans': "MS Dhoni, whose full name is Mahendra Singh Dhoni, is an Indian cricketer who played for the Indian national team. He was born on July 7, 1981, in a small village in the state of Jharkhand. Dhoni is widely regarded as one of the greatest wicket-keepers in the history of cricket.\n\nDhoni's cricket career began at a young age, and he quickly rose through the ranks of the Indian cricket system. He made his international debut in 2004 and quickly became a key player for India, known for his exceptional skills behind the stumps and his ability to score crucial runs.\n\nOne of Dhoni's most notable achievements was his role in India's 2007 T20 World Cup and 2011 Cricket World Cup victories. He was also the captain of the Indian team that won the 2010 and 2013 ICC World Twenty20 tournaments.\n\nDhoni retired from international cricket in 2014, but he continued to play for his state team, the Tamil Nadu Cricket Association, and for the Chennai Super Kin